In [8]:
import pandas as pd

df_sample = pd.read_csv("../data/complaints_raw.csv", nrows=5)
print(df_sample.shape)
df_sample.head()

(5, 16)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-07-07T21:03:24.000Z,Debt collection,Medical debt,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,"XX/XX/2023, MEDICAL DATA SYSTEMS I put a colle...",NaN,"Medical Data Systems, Inc.",PA,19138,NaN,Web,2023-07-07T21:09:04.000Z,Closed with explanation,Yes,7220414
1,2023-07-07T01:28:01.000Z,Debt collection,Mortgage debt,Took or threatened to take negative or legal a...,Threatened or suggested your credit would be d...,Paid In Full! XXXX & XXXX XXXX XXXX XXXX XXX...,Company has responded to the consumer and the ...,"PAID IN FULL, INC.",AZ,852XX,NaN,Web,2023-07-07T02:16:09.000Z,Closed with explanation,Yes,7212581
2,2023-07-07T01:08:41.000Z,Debt collection,Other debt,False statements or representation,Attempted to collect wrong amount,XXXX XXXX was paid in full on XX/XX/2023 ( see...,NaN,First Credit Services Inc.,AZ,85381,NaN,Web,2023-08-01T17:05:33.000Z,Closed with explanation,Yes,7212612
3,2023-07-07T02:34:58.000Z,Debt collection,Credit card debt,Written notification about debt,Didn't receive enough information to verify debt,This company is sending me information I alrea...,NaN,"Portfolio Recovery Associates, LLC",FL,33604,NaN,Web,2023-07-07T02:44:34.000Z,Closed with explanation,Yes,7212437
4,2023-07-07T01:29:33.000Z,Debt collection,I do not know,Written notification about debt,Didn't receive notice of right to dispute,Spring Oaks Capital LLC called my mother in la...,NaN,"Spring Oaks Capital, LLC",NC,27105,NaN,Web,2023-07-07T01:47:46.000Z,Closed with explanation,Yes,7212703


In [9]:
import pandas as pd

# Specify dtypes upfront to reduce memory footprint
dtype_map = {
    "Product": "category",
    "Sub-product": "category",
    "Issue": "category",
    "Sub-issue": "category",
    "Company public response": "category",
    "Company": "category",
    "State": "category",
    "Submitted via": "category",
    "Company response to consumer": "category",
    "Timely response?": "category",
    "Tags": "category",
    "Complaint ID": "int64",
}

df = pd.read_csv(
    "../data/complaints_raw.csv",
    dtype=dtype_map,
    parse_dates=["Date received", "Date sent to company"],
)

print(df.shape)
print(df.memory_usage(deep=True).sum() / 1e6, "MB")
df.info()

(878515, 16)
442.479252 MB
<class 'pandas.DataFrame'>
RangeIndex: 878515 entries, 0 to 878514
Data columns (total 16 columns):
 #   Column                        Non-Null Count   Dtype              
---  ------                        --------------   -----              
 0   Date received                 878515 non-null  datetime64[us, UTC]
 1   Product                       878515 non-null  category           
 2   Sub-product                   878515 non-null  category           
 3   Issue                         878515 non-null  category           
 4   Sub-issue                     878291 non-null  category           
 5   Consumer complaint narrative  321175 non-null  str                
 6   Company public response       319123 non-null  category           
 7   Company                       878515 non-null  category           
 8   State                         876622 non-null  category           
 9   ZIP code                      878273 non-null  str                
 10  Tags

In [10]:
print(df["Product"].value_counts())
print()
print(df["Timely response?"].value_counts())
print()
print("Narrative available by Product:")
print(df.groupby("Product", observed=True)["Consumer complaint narrative"].apply(lambda x: x.notna().mean()))

Product
Debt collection    643817
Credit card        234698
Name: count, dtype: int64

Timely response?
Yes    859512
No      19003
Name: count, dtype: int64

Narrative available by Product:
Product
Credit card        0.444333
Debt collection    0.336883
Name: Consumer complaint narrative, dtype: float64


In [11]:
late = df[df["Timely response?"] == "No"]
print("Late responses by Product:")
print(late["Product"].value_counts(normalize=True))
print()
print("Late responses by Issue (top 10):")
print(late["Issue"].value_counts(normalize=True).head(10))
print()
print("Overall late rate by Product:")
print(df.groupby("Product", observed=True)["Timely response?"].apply(lambda x: (x == "No").mean()))

Late responses by Product:
Product
Debt collection    0.944482
Credit card        0.055518
Name: proportion, dtype: float64

Late responses by Issue (top 10):
Issue
Attempts to collect debt not owed                                0.327527
Took or threatened to take negative or legal action              0.297111
Written notification about debt                                  0.134873
False statements or representation                               0.120244
Communication tactics                                            0.039310
Problem with a purchase shown on your statement                  0.017524
Threatened to contact someone or share information improperly    0.013314
Electronic communications                                        0.012103
Other features, terms, or problems                               0.006999
Getting a credit card                                            0.006525
Name: proportion, dtype: float64

Overall late rate by Product:
Product
Credit card        0.00

In [12]:
df["year_month"] = df["Date received"].dt.to_period("M")

trend = df.groupby(["year_month", "Product"], observed=True)["Timely response?"].apply(
    lambda x: (x == "No").mean()
).unstack()

print(trend.tail(12))  # last 12 months

Product     Credit card  Debt collection
year_month                              
2025-08        0.003157         0.029149
2025-09        0.002234         0.021442
2025-10        0.002967         0.028222
2025-11        0.006410         0.031660
2025-12        0.012966         0.027835
2026-01        0.011768         0.029967
2026-02        0.017980         0.032412
2026-03        0.013359         0.033999
2026-04        0.013306         0.042806
2026-05        0.005310         0.051049
2026-06        0.004268         0.035221
2026-07        0.000000         0.000000


In [13]:
df = df[df["year_month"] != "2026-07"].copy()
print(df.shape)

(877457, 17)


In [14]:
import numpy as np

# 1. Days between complaint received and sent to company — proxy for internal handling delay
df["days_to_send"] = (df["Date sent to company"] - df["Date received"]).dt.days

# 2. Narrative availability as its own feature (not just a filter)
df["has_narrative"] = df["Consumer complaint narrative"].notna().astype(int)

# 3. Narrative length (0 if missing) — cheap urgency proxy
df["narrative_length"] = df["Consumer complaint narrative"].fillna("").str.len()

# 4. High-risk Issue flag, based on what you just found drives lateness
high_risk_issues = [
    "Attempts to collect debt not owed",
    "Took or threatened to take negative or legal action"
]
df["high_risk_issue"] = df["Issue"].isin(high_risk_issues).astype(int)

# 5. Month, to let a model capture the drift you just found
df["received_month"] = df["Date received"].dt.month

print(df[["days_to_send", "has_narrative", "narrative_length", "high_risk_issue", "received_month"]].describe(include="all"))

        days_to_send  has_narrative  narrative_length  high_risk_issue  \
count  877457.000000  877457.000000     877457.000000    877457.000000   
mean        1.369645       0.366029        375.427230         0.439053   
std         7.554900       0.481718        892.699196         0.496272   
min         0.000000       0.000000          0.000000         0.000000   
25%         0.000000       0.000000          0.000000         0.000000   
50%         0.000000       0.000000          0.000000         0.000000   
75%         0.000000       1.000000        370.000000         1.000000   
max       445.000000       1.000000      32785.000000         1.000000   

       received_month  
count   877457.000000  
mean         6.210317  
std          3.485308  
min          1.000000  
25%          3.000000  
50%          6.000000  
75%          9.000000  
max         12.000000  


In [15]:
import numpy as np

print(df["narrative_length"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print()
print("days_to_send percentiles:")
print(df["days_to_send"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

count    877457.000000
mean        375.427230
std         892.699196
min           0.000000
50%           0.000000
75%         370.000000
90%        1249.000000
95%        1896.000000
99%        3752.000000
max       32785.000000
Name: narrative_length, dtype: float64

days_to_send percentiles:
count    877457.000000
mean          1.369645
std           7.554900
min           0.000000
50%           0.000000
90%           0.000000
95%           5.000000
99%          42.000000
max         445.000000
Name: days_to_send, dtype: float64


In [16]:
df["days_to_send_capped"] = df["days_to_send"].clip(upper=42)

In [17]:
# Subset 1: structured-only baseline (full dataset, no text dependency)
df_structured = df.copy()

# Subset 2: narrative-present rows only (for NLP pipeline)
df_narrative = df[df["has_narrative"] == 1].copy()

print("Structured (full):", df_structured.shape)
print("Narrative subset:", df_narrative.shape)

# Sort both by date — required before any time-based split
df_structured = df_structured.sort_values("Date received").reset_index(drop=True)
df_narrative = df_narrative.sort_values("Date received").reset_index(drop=True)

print(df_structured["Date received"].min(), "to", df_structured["Date received"].max())

Structured (full): (877457, 23)
Narrative subset: (321175, 23)
2023-07-07 00:03:51+00:00 to 2026-06-30 23:59:01+00:00


In [18]:
# Binary target: 1 = late (minority, business-critical), 0 = on time
df_structured["target_late"] = (df_structured["Timely response?"] == "No").astype(int)

print(df_structured["target_late"].value_counts(normalize=True))

# Time-based split: train on earlier ~80%, test on most recent ~20%
split_idx = int(len(df_structured) * 0.8)
train_structured = df_structured.iloc[:split_idx]
test_structured = df_structured.iloc[split_idx:]

print("Train date range:", train_structured["Date received"].min(), "to", train_structured["Date received"].max())
print("Test date range:", test_structured["Date received"].min(), "to", test_structured["Date received"].max())
print("Train late rate:", train_structured["target_late"].mean())
print("Test late rate:", test_structured["target_late"].mean())

target_late
0    0.978343
1    0.021657
Name: proportion, dtype: float64
Train date range: 2023-07-07 00:03:51+00:00 to 2026-02-02 19:52:05+00:00
Test date range: 2026-02-02 19:52:53+00:00 to 2026-06-30 23:59:01+00:00
Train late rate: 0.018758770024146502
Test late rate: 0.03324937888906617


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

feature_cols = ["Product", "Sub-product", "Issue", "high_risk_issue", 
                 "days_to_send_capped", "has_narrative", "received_month", "State"]

X_train = train_structured[feature_cols]
y_train = train_structured["target_late"]
X_test = test_structured[feature_cols]
y_test = test_structured["target_late"]

categorical_cols = ["Product", "Sub-product", "Issue", "State"]
numeric_cols = ["high_risk_issue", "days_to_send_capped", "has_narrative", "received_month"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
], remainder="passthrough")

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipeline.fit(X_train, y_train)
print("Trained.")

Trained.


In [20]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["On time", "Late"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC (average precision):", average_precision_score(y_test, y_proba))

              precision    recall  f1-score   support

     On time       0.99      0.39      0.56    169657
        Late       0.05      0.85      0.09      5835

    accuracy                           0.41    175492
   macro avg       0.52      0.62      0.32    175492
weighted avg       0.96      0.41      0.54    175492

ROC-AUC: 0.7070378194738485
PR-AUC (average precision): 0.07187572970789845


In [21]:
import lightgbm as lgb

In [22]:
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# LightGBM can take category dtype columns directly — no one-hot needed
train_lgb = train_structured[feature_cols].copy()
test_lgb = test_structured[feature_cols].copy()

for col in categorical_cols:
    train_lgb[col] = train_lgb[col].astype("category")
    test_lgb[col] = test_lgb[col].astype("category")

# scale_pos_weight = ratio of negative to positive class, LightGBM's imbalance handling
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", pos_weight)

lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=pos_weight,
    random_state=42
)

lgb_model.fit(
    train_lgb, y_train,
    categorical_feature=categorical_cols
)

y_pred_lgb = lgb_model.predict(test_lgb)
y_proba_lgb = lgb_model.predict_proba(test_lgb)[:, 1]

print(classification_report(y_test, y_pred_lgb, target_names=["On time", "Late"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgb))
print("PR-AUC:", average_precision_score(y_test, y_proba_lgb))

scale_pos_weight: 52.30839914945322
[LightGBM] [Info] Number of positive: 13168, number of negative: 688797
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022225 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 157
[LightGBM] [Info] Number of data points in the train set: 701965, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.018759 -> initscore=-3.957157
[LightGBM] [Info] Start training from score -3.957157
              precision    recall  f1-score   support

     On time       0.99      0.42      0.59    169657
        Late       0.05      0.82      0.09      5835

    accuracy                           0.43    175492
   macro avg       0.52      0.62      0.34    175492
weighted avg       0.95      0.43      0.57    175492

ROC-AUC: 0.6903926314476966
PR-AUC: 0.06793217917467408


In [23]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

param_dist = {
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [3, 5, 7, -1],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators": [100, 300, 500],
    "min_child_samples": [10, 20, 50],
}

lgb_base = lgb.LGBMClassifier(scale_pos_weight=pos_weight, random_state=42, verbose=-1)

search = RandomizedSearchCV(
    lgb_base, param_dist, n_iter=20, scoring="average_precision",
    cv=3, random_state=42, n_jobs=-1
)

search.fit(train_lgb, y_train, categorical_feature=categorical_cols)

print("Best params:", search.best_params_)
print("Best CV PR-AUC:", search.best_score_)

Best params: {'num_leaves': 63, 'n_estimators': 300, 'min_child_samples': 50, 'max_depth': 3, 'learning_rate': 0.01}
Best CV PR-AUC: 0.03286080468366149


In [24]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    lgb_base, param_dist, n_iter=20, scoring="average_precision",
    cv=tscv, random_state=42, n_jobs=-1, verbose=2
)

search.fit(train_lgb, y_train, categorical_feature=categorical_cols)

print("Best params:", search.best_params_)
print("Best CV PR-AUC:", search.best_score_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'num_leaves': 63, 'n_estimators': 300, 'min_child_samples': 50, 'max_depth': 3, 'learning_rate': 0.01}
Best CV PR-AUC: 0.04602141448207203


In [25]:
best_model = search.best_estimator_
y_pred_tuned = best_model.predict(test_lgb)
y_proba_tuned = best_model.predict_proba(test_lgb)[:, 1]

print(classification_report(y_test, y_pred_tuned, target_names=["On time", "Late"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))
print("PR-AUC:", average_precision_score(y_test, y_proba_tuned))

              precision    recall  f1-score   support

     On time       0.99      0.44      0.61    169657
        Late       0.05      0.82      0.09      5835

    accuracy                           0.45    175492
   macro avg       0.52      0.63      0.35    175492
weighted avg       0.96      0.45      0.59    175492

ROC-AUC: 0.7111336720468804
PR-AUC: 0.07369267339777362


In [26]:
import sentence_transformers
print(sentence_transformers.__version__)

5.6.0


In [27]:
from sentence_transformers import SentenceTransformer
import time

model = SentenceTransformer("all-MiniLM-L6-v2")
narratives = df_narrative["Consumer complaint narrative"].fillna("").tolist()

sample = narratives[:200]
start = time.time()
_ = model.encode(sample, batch_size=64, show_progress_bar=False)
print(f"{time.time() - start:.2f}s for 200 docs")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

117.06s for 200 docs


In [28]:
import torch
print(torch.get_num_threads())
print(torch.__config__.parallel_info())

2
ATen/Parallel:
	at::get_num_threads() : 2
	at::get_num_interop_threads() : 2
OpenMP 2019
	omp_get_max_threads() : 2
Intel(R) oneAPI Math Kernel Library Version 2026.0-Product Build 20260401 for Intel(R) 64 architecture applications
	mkl_get_max_threads() : 2
Intel(R) MKL-DNN v3.11.2 (Git Hash 03c022d3ffdcee958cfacbe720048e725fdf644c)
std::thread::hardware_concurrency() : 4
Environment variables:
	OMP_NUM_THREADS : [not set]
	MKL_NUM_THREADS : [not set]
ATen parallel backend: OpenMP



In [29]:
torch.set_num_threads(4)

In [30]:
from sklearn.model_selection import train_test_split

# Sample 30,000 narratives, stratified by Product to preserve the ~2.7:1 ratio
df_narrative_sample, _ = train_test_split(
    df_narrative,
    train_size=30000,
    stratify=df_narrative["Product"],
    random_state=42
)

df_narrative_sample = df_narrative_sample.sort_values("Date received").reset_index(drop=True)

print(df_narrative_sample["Product"].value_counts())
print(df_narrative_sample.shape)

Product
Debt collection    20259
Credit card         9741
Name: count, dtype: int64
(30000, 23)


In [31]:
import numpy as np
embeddings = np.load("../data/narrative_embeddings_sample.npy")
print(embeddings.shape)
print(len(df_narrative_sample))
assert embeddings.shape[0] == len(df_narrative_sample), "Mismatch! Do not proceed until fixed."
print("Aligned correctly.")

(30000, 384)
30000
Aligned correctly.


In [32]:
import bertopic
print(bertopic.__version__)

0.17.4


In [33]:
from bertopic import BERTopic

narratives_sample = df_narrative_sample["Consumer complaint narrative"].fillna("").tolist()

topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(narratives_sample, embeddings=embeddings)

print(topic_model.get_topic_info().head(20))

2026-07-09 13:29:38,729 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-09 13:33:53,734 - BERTopic - Dimensionality - Completed ✓
2026-07-09 13:33:53,952 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-09 13:34:16,757 - BERTopic - Cluster - Completed ✓
2026-07-09 13:34:16,998 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-09 13:34:38,457 - BERTopic - Representation - Completed ✓


    Topic  Count                                             Name  \
0      -1  11580                              -1_xxxx_was_the_and   
1       0    681                       0_capital_one_card_capitol   
2       1    507                     1_lvnv_funding_llc_resurgent   
3       2    470                        2_calling_calls_call_stop   
4       3    458                    3_midland_management_mcm_debt   
5       4    363               4_collection_validation_fdcpa_debt   
6       5    340                  5_jefferson_systems_capital_llc   
7       6    329               6_debt_validation_alleged_original   
8       7    293                  7_lease_apartment_rent_property   
9       8    260                        8_wells_fargo_card_fargos   
10      9    244             9_fcra_inaccurate_reporting_accuracy   
11     10    240             10_portfolio_recovery_pra_associates   
12     11    239               11_identity_theft_victim_knowledge   
13     12    233               12_

In [34]:
topic_model.save("../data/bertopic_model", serialization="safetensors", save_ctfidf=True)
print(topic_model.get_topic_info().shape)

(404, 5)


In [35]:
topic_model.reduce_topics(narratives_sample, nr_topics=40)

reduced_info = topic_model.get_topic_info()
print(reduced_info.shape)
print(reduced_info.head(40))

2026-07-09 13:39:56,107 - BERTopic - Topic reduction - Reducing number of topics
2026-07-09 13:39:56,697 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-09 13:40:28,172 - BERTopic - Representation - Completed ✓
2026-07-09 13:40:28,499 - BERTopic - Topic reduction - Reduced number of topics from 404 to 40


(40, 5)
    Topic  Count                                            Name  \
0      -1  11580                              -1_xxxx_the_to_and   
1       0   4058                              0_debt_of_the_this   
2       1   2546                             1_the_xxxx_was_card   
3       2   2031               2_accounts_accurate_report_review   
4       3   1555                              3_xxxx_they_to_and   
5       4   1132             4_accounts_fraudulent_report_credit   
6       5    919                     5_falsely_midland_credit_is   
7       6    901                          6_one_capital_the_card   
8       7    837                      7_late_payment_payments_my   
9       8    735           8_these_transactions_report_authorize   
10      9    686                          9_debt_1099c_this_xxxx   
11     10    450                    10_payment_interest_fee_late   
12     11    404                  11_consumer_information_15_usc   
13     12    344                        

In [36]:
print("Topic 0 samples:")
print(topic_model.get_representative_docs(0)[:3])
print()
print("Topic 15 samples:")
print(topic_model.get_representative_docs(15)[:3])
print()
print("Topic 18 samples:")
print(topic_model.get_representative_docs(18)[:3])

Topic 0 samples:
['This letter was sent out on XX/XX/year>. \n\nTo : ATTN : XXXX XXXX XXXX XXXX XXXX XXXX XXXXXXXX XXXX. XXXX XXXX XXXX, MI XXXX RE : Request for Debt Validation Under the Fair Debt Collection Practices Act To Whom It May Concern : I am writing in response to a letter sent about an alleged debt and/or a collection attempt from your company. This is a formal request for validation of the alleged debt as provided for by the Fair Debt Collection Practices Act ( FDCPA ), 15 U.S.C. 1692g.\n\nPlease provide the following : 1. The name and address of the original creditor 2. A copy of the original signed contract or agreement bearing my signature 3. A complete accounting and payment history of the alleged debt 4. Proof that your company is authorized to collect on this debt, including the chain of assignment and ownership 5. The amount of the original debt, any added fees, interest, or payments Until this information is provided, I request that you cease all collection activit

In [37]:
dup_counts = df_narrative_sample["Consumer complaint narrative"].value_counts()
print("Narratives appearing more than once:")
print(dup_counts[dup_counts > 1].head(20))
print()
print(f"Total duplicate narrative instances: {(dup_counts[dup_counts > 1].sum())}")
print(f"Percentage of sample that are duplicates: {(dup_counts[dup_counts > 1].sum() / len(df_narrative_sample)) * 100:.1f}%")

Narratives appearing more than once:
Consumer complaint narrative
There are collection accounts on my report that I believe contain inaccurate information. Under my rights pursuant to 15 USC 1681e ( b ) and 15 USC 1681i, I am entitled to an accurate credit report. I request a review of these entries, and if they can not be verified as accurate, I ask that they be removed.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [38]:
# Check unique narrative count before deduping
print("Total rows:", len(df_narrative_sample))
print("Unique narratives:", df_narrative_sample["Consumer complaint narrative"].nunique())

# Deduplicate, keeping first occurrence
df_narrative_dedup = df_narrative_sample.drop_duplicates(subset="Consumer complaint narrative", keep="first").copy()
print("After dedup:", df_narrative_dedup.shape)

# Check Product balance held up
print(df_narrative_dedup["Product"].value_counts())

Total rows: 30000
Unique narratives: 25377
After dedup: (25377, 23)
Product
Debt collection    16153
Credit card         9224
Name: count, dtype: int64


In [39]:
# Get the positional indices of rows that survived dedup, in the ORIGINAL df_narrative_sample order
kept_mask = df_narrative_sample["Consumer complaint narrative"].duplicated(keep="first") == False
kept_positions = np.where(kept_mask.values)[0]

embeddings_dedup = embeddings[kept_positions]

print(embeddings_dedup.shape)
print(len(df_narrative_dedup))
assert embeddings_dedup.shape[0] == len(df_narrative_dedup), "Mismatch — do not proceed"
print("Aligned correctly.")

# Save the deduped embeddings so this alignment work isn't repeated
np.save("../data/narrative_embeddings_dedup.npy", embeddings_dedup)

(25377, 384)
25377
Aligned correctly.


In [40]:
from bertopic import BERTopic

narratives_dedup = df_narrative_dedup["Consumer complaint narrative"].fillna("").tolist()

topic_model_dedup = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    verbose=True
)

topics_dedup, probs_dedup = topic_model_dedup.fit_transform(narratives_dedup, embeddings=embeddings_dedup)

print(topic_model_dedup.get_topic_info().shape)
print(topic_model_dedup.get_topic_info().head(20))

2026-07-09 13:47:16,178 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-09 13:48:04,534 - BERTopic - Dimensionality - Completed ✓
2026-07-09 13:48:04,717 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-09 13:48:29,017 - BERTopic - Cluster - Completed ✓
2026-07-09 13:48:29,241 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-09 13:48:45,201 - BERTopic - Representation - Completed ✓


(254, 5)
    Topic  Count                                      Name  \
0      -1  11096                        -1_xxxx_the_and_to   
1       0    683                    0_capital_one_card_was   
2       1    539                1_calls_calling_call_phone   
3       2    528                2_chase_card_points_chases   
4       3    510              3_lvnv_funding_llc_resurgent   
5       4    448             4_midland_management_mcm_debt   
6       5    343          5_jefferson_capital_systems_debt   
7       6    324           6_lease_apartment_rent_property   
8       7    279             7_letter_validation_sent_debt   
9       8    258            8_debt_you_validation_original   
10      9    257                    9_wells_fargo_card_was   
11     10    249      10_portfolio_recovery_associates_pra   
12     11    249        11_insurance_bill_medical_hospital   
13     12    231             12_synchrony_bank_amazon_card   
14     13    223       13_discover_discovers_card_merchant   

In [41]:
topic_model_dedup.save("../data/bertopic_model_dedup", serialization="safetensors", save_ctfidf=True)

In [42]:
print(topic_model_dedup.get_topic_info().shape[0])

254


In [43]:
topic_model_dedup.reduce_topics(narratives_dedup, nr_topics=40)

reduced_info_dedup = topic_model_dedup.get_topic_info()
print(reduced_info_dedup.shape)
print(reduced_info_dedup.head(40))

2026-07-09 14:58:14,107 - BERTopic - Topic reduction - Reducing number of topics
2026-07-09 14:58:14,811 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-09 14:58:38,306 - BERTopic - Representation - Completed ✓
2026-07-09 14:58:38,577 - BERTopic - Topic reduction - Reduced number of topics from 254 to 40


(40, 5)
    Topic  Count                                           Name  \
0      -1  11096                             -1_xxxx_the_to_and   
1       0   4138                             0_debt_of_the_this   
2       1   3281                             1_the_xxxx_card_to   
3       2   1846                             2_xxxx_to_and_they   
4       3   1338                  3_xxxx_report_credit_accounts   
5       4    721                           4_capital_one_the_my   
6       5    549                     5_late_payment_payments_my   
7       6    525                 6_interest_balance_payment_was   
8       7    301                  7_consumer_information_15_usc   
9       8    246                8_falsely_reporting_is_research   
10      9    164         9_credit_inquiries_application_inquiry   
11     10    140                          10_1099c_irs_off_form   
12     11    138                   11_identity_theft_my_account   
13     12     92                    12_hardship_limit_

In [44]:
print(reduced_info_dedup[reduced_info_dedup["Topic"] == -1])

   Topic  Count                Name  \
0     -1  11096  -1_xxxx_the_to_and   

                                      Representation  \
0  [xxxx, the, to, and, of, my, this, that, credi...   

                                 Representative_Docs  
0  [Re : Disputing error [ s ] on credit report T...  


In [45]:
df_narrative_dedup["topic"] = topic_model_dedup.topics_
df_narrative_dedup["topic_name"] = df_narrative_dedup["topic"].map(
    reduced_info_dedup.set_index("Topic")["Name"]
)

print(df_narrative_dedup[["Product", "Issue", "topic", "topic_name"]].head(10))

           Product                                              Issue  topic  \
0  Debt collection                    Written notification about debt     -1   
1  Debt collection                  Attempts to collect debt not owed      0   
2  Debt collection                  Attempts to collect debt not owed      0   
3  Debt collection                              Communication tactics     -1   
4  Debt collection                  Attempts to collect debt not owed      0   
5  Debt collection                    Written notification about debt     -1   
6  Debt collection                    Written notification about debt     -1   
7  Debt collection                  Attempts to collect debt not owed     -1   
8  Debt collection                  Attempts to collect debt not owed     11   
9  Debt collection  Took or threatened to take negative or legal a...     14   

                          topic_name  
0                 -1_xxxx_the_to_and  
1                 0_debt_of_the_this  
2 

In [46]:
topic_model_dedup.save("../data/bertopic_model_dedup", serialization="safetensors", save_ctfidf=True)
df_narrative_dedup.to_csv("../data/narrative_dedup_with_topics.csv", index=False)
print("Saved.")

Saved.


In [47]:
df_narrative_dedup["target_late"] = (df_narrative_dedup["Timely response?"] == "No").astype(int)
df_narrative_dedup = df_narrative_dedup.sort_values("Date received").reset_index(drop=True)

print(df_narrative_dedup["target_late"].value_counts(normalize=True))

split_idx_narr = int(len(df_narrative_dedup) * 0.8)
train_narr = df_narrative_dedup.iloc[:split_idx_narr]
test_narr = df_narrative_dedup.iloc[split_idx_narr:]

print("Train late rate:", train_narr["target_late"].mean())
print("Test late rate:", test_narr["target_late"].mean())

target_late
0    0.980534
1    0.019466
Name: proportion, dtype: float64
Train late rate: 0.01778237525245062
Test late rate: 0.02620173364854216


In [48]:
feature_cols_narr = ["Product", "Sub-product", "Issue", "high_risk_issue",
                      "days_to_send_capped", "has_narrative", "received_month",
                      "State", "topic"]

categorical_cols_narr = ["Product", "Sub-product", "Issue", "State", "topic"]

X_train_narr = train_narr[feature_cols_narr].copy()
X_test_narr = test_narr[feature_cols_narr].copy()
y_train_narr = train_narr["target_late"]
y_test_narr = test_narr["target_late"]

for col in categorical_cols_narr:
    X_train_narr[col] = X_train_narr[col].astype("category")
    X_test_narr[col] = X_test_narr[col].astype("category")

pos_weight_narr = (y_train_narr == 0).sum() / (y_train_narr == 1).sum()

lgb_narr = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=pos_weight_narr, random_state=42
)

lgb_narr.fit(X_train_narr, y_train_narr, categorical_feature=categorical_cols_narr)

y_pred_narr = lgb_narr.predict(X_test_narr)
y_proba_narr = lgb_narr.predict_proba(X_test_narr)[:, 1]

print(classification_report(y_test_narr, y_pred_narr, target_names=["On time", "Late"]))
print("ROC-AUC:", roc_auc_score(y_test_narr, y_proba_narr))
print("PR-AUC:", average_precision_score(y_test_narr, y_proba_narr))

              precision    recall  f1-score   support

     On time       0.98      0.91      0.94      4943
        Late       0.05      0.18      0.08       133

    accuracy                           0.89      5076
   macro avg       0.51      0.54      0.51      5076
weighted avg       0.95      0.89      0.92      5076

ROC-AUC: 0.6253310293739608
PR-AUC: 0.051564571715363244


In [49]:
feature_cols_no_topic = ["Product", "Sub-product", "Issue", "high_risk_issue",
                          "days_to_send_capped", "has_narrative", "received_month", "State"]
categorical_cols_no_topic = ["Product", "Sub-product", "Issue", "State"]

X_train_nt = train_narr[feature_cols_no_topic].copy()
X_test_nt = test_narr[feature_cols_no_topic].copy()

for col in categorical_cols_no_topic:
    X_train_nt[col] = X_train_nt[col].astype("category")
    X_test_nt[col] = X_test_nt[col].astype("category")

lgb_no_topic = lgb.LGBMClassifier(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    scale_pos_weight=pos_weight_narr, random_state=42
)
lgb_no_topic.fit(X_train_nt, y_train_narr, categorical_feature=categorical_cols_no_topic)

y_proba_nt = lgb_no_topic.predict_proba(X_test_nt)[:, 1]
print("PR-AUC without topic:", average_precision_score(y_test_narr, y_proba_nt))
print("ROC-AUC without topic:", roc_auc_score(y_test_narr, y_proba_nt))

PR-AUC without topic: 0.04778248605763391
ROC-AUC without topic: 0.6053445367414084


In [52]:
print(df_narrative_dedup["Date received"].is_monotonic_increasing)

True


In [53]:
# Reload from disk, since in-memory df_narrative_dedup is already sorted (order info lost)
import pandas as pd
import numpy as np

df_narrative_dedup_fresh = pd.read_csv("../data/narrative_dedup_with_topics.csv", parse_dates=["Date received", "Date sent to company"])
embeddings_dedup_fresh = np.load("../data/narrative_embeddings_dedup.npy")

print(len(df_narrative_dedup_fresh), embeddings_dedup_fresh.shape)
assert len(df_narrative_dedup_fresh) == embeddings_dedup_fresh.shape[0], "Mismatch on reload!"
print("Reload aligned correctly.")

# Now sort BOTH together, correctly, once
sort_order = df_narrative_dedup_fresh["Date received"].argsort().values
df_narrative_dedup = df_narrative_dedup_fresh.iloc[sort_order].reset_index(drop=True)
embeddings_dedup_sorted = embeddings_dedup_fresh[sort_order]

print(df_narrative_dedup["Date received"].is_monotonic_increasing)
print(embeddings_dedup_sorted.shape, len(df_narrative_dedup))

25377 (25377, 384)
Reload aligned correctly.
True
(25377, 384) 25377


In [54]:
df_narrative_dedup["target_late"] = (df_narrative_dedup["Timely response?"] == "No").astype(int)

split_idx_narr = int(len(df_narrative_dedup) * 0.8)
train_narr = df_narrative_dedup.iloc[:split_idx_narr]
test_narr = df_narrative_dedup.iloc[split_idx_narr:]
train_emb = embeddings_dedup_sorted[:split_idx_narr]
test_emb = embeddings_dedup_sorted[split_idx_narr:]

print("Train late rate:", train_narr["target_late"].mean())
print("Test late rate:", test_narr["target_late"].mean())

Train late rate: 0.01778237525245062
Test late rate: 0.02620173364854216


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split as tts
from sklearn.metrics import classification_report as cr

y_product = df_narrative_dedup["Product"]

X_train_emb, X_test_emb, y_train_prod, y_test_prod = tts(
    embeddings_dedup_sorted, y_product, test_size=0.2, stratify=y_product, random_state=42
)

clf_emb = LogisticRegression(max_iter=1000)
clf_emb.fit(X_train_emb, y_train_prod)

y_pred_prod = clf_emb.predict(X_test_emb)
print(cr(y_test_prod, y_pred_prod))

                 precision    recall  f1-score   support

    Credit card       0.92      0.88      0.90      1845
Debt collection       0.93      0.95      0.94      3231

       accuracy                           0.93      5076
      macro avg       0.92      0.92      0.92      5076
   weighted avg       0.93      0.93      0.93      5076



In [56]:
y_issue = df_narrative_dedup["Issue"]
print(y_issue.value_counts())

Issue
Attempts to collect debt not owed                                  7490
Written notification about debt                                    3104
Problem with a purchase shown on your statement                    2558
False statements or representation                                 2333
Took or threatened to take negative or legal action                1511
Fees or interest                                                   1107
Other features, terms, or problems                                 1087
Getting a credit card                                              1067
Communication tactics                                              1061
Closing your account                                                717
Problem when making payments                                        562
Incorrect information on your report                                497
Advertising and marketing, including promotional offers             473
Problem with a company's investigation into an existing pr

In [57]:
top_issues = y_issue.value_counts()
keep_issues = top_issues[top_issues >= 400].index.tolist()
print(f"Keeping {len(keep_issues)} categories:", keep_issues)

df_narrative_dedup["issue_grouped"] = df_narrative_dedup["Issue"].apply(
    lambda x: x if x in keep_issues else "Other"
)
print(df_narrative_dedup["issue_grouped"].value_counts())

Keeping 15 categories: ['Attempts to collect debt not owed', 'Written notification about debt', 'Problem with a purchase shown on your statement', 'False statements or representation', 'Took or threatened to take negative or legal action', 'Fees or interest', 'Other features, terms, or problems', 'Getting a credit card', 'Communication tactics', 'Closing your account', 'Problem when making payments', 'Incorrect information on your report', 'Advertising and marketing, including promotional offers', "Problem with a company's investigation into an existing problem", 'Trouble using your card']
issue_grouped
Attempts to collect debt not owed                                  7490
Written notification about debt                                    3104
Problem with a purchase shown on your statement                    2558
False statements or representation                                 2333
Took or threatened to take negative or legal action                1511
Fees or interest             

In [58]:
y_issue_grouped = df_narrative_dedup["issue_grouped"]

X_train_emb2, X_test_emb2, y_train_issue, y_test_issue = tts(
    embeddings_dedup_sorted, y_issue_grouped, test_size=0.2, stratify=y_issue_grouped, random_state=42
)

clf_issue = LogisticRegression(max_iter=1000, class_weight="balanced")
clf_issue.fit(X_train_emb2, y_train_issue)

y_pred_issue = clf_issue.predict(X_test_emb2)
print(cr(y_test_issue, y_pred_issue))

                                                                 precision    recall  f1-score   support

        Advertising and marketing, including promotional offers       0.49      0.66      0.56        95
                              Attempts to collect debt not owed       0.71      0.33      0.45      1498
                                           Closing your account       0.55      0.71      0.62       143
                                          Communication tactics       0.50      0.62      0.56       212
                             False statements or representation       0.34      0.48      0.40       467
                                               Fees or interest       0.64      0.61      0.62       222
                                          Getting a credit card       0.43      0.62      0.51       213
                           Incorrect information on your report       0.19      0.43      0.26        99
                                                      

In [59]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test_issue, y_pred_issue, labels=clf_issue.classes_)
cm_df = pd.DataFrame(cm, index=clf_issue.classes_, columns=clf_issue.classes_)
print(cm_df.loc["Attempts to collect debt not owed"].sort_values(ascending=False))

Attempts to collect debt not owed                                  498
False statements or representation                                 260
Written notification about debt                                    220
Took or threatened to take negative or legal action                141
Getting a credit card                                               87
Other                                                               86
Incorrect information on your report                                80
Communication tactics                                               43
Problem with a purchase shown on your statement                     25
Closing your account                                                15
Problem when making payments                                        14
Other features, terms, or problems                                   8
Problem with a company's investigation into an existing problem      8
Advertising and marketing, including promotional offers              7
Fees o

In [60]:
import joblib

joblib.dump(lgb_model, "../data/lgb_model_structured.pkl")
joblib.dump(search.best_estimator_, "../data/lgb_model_tuned.pkl")
joblib.dump(clf_issue, "../data/clf_issue_embeddings.pkl")
print("Models saved.")

Models saved.
